# 7B. Stronger RGB Backbone Colab

This notebook extends Stage 7 by replacing the frozen RGB feature extractor with a stronger backbone while keeping the counting task and the exercise subset fixed.


## Reason, Approach, Result Interpretation

**Reason**
- Stage 7 showed that RGB helps on `push_up`, but not uniformly.
- The next question is whether the limitation is the visual representation itself rather than RGB as a category.

**Approach**
- Keep the same three exercises: `squat`, `pull_up`, and `push_up`.
- Keep the same counting formulation: per-exercise TCN regression.
- Replace the frozen ResNet18 extractor with a stronger frozen ResNet50 extractor.
- Use more conservative extraction settings to stay within Colab resource limits.
- Compare the stronger RGB results against both pose `6B` and the Stage 7 RGB baseline.

**How to interpret the result**
- If the stronger backbone improves Stage 7 RGB consistently, the next bottleneck was likely representation strength.
- If it does not improve much, the next step should be multimodal or a different temporal model rather than a larger RGB encoder alone.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Environment Setup

**Why this section exists**
- The stronger RGB branch uses the same Stage 7 scripts, but with a different extractor backbone and a separate output location.

**Approach**
- Resolve project paths in Drive.
- Sync the current repo copies of the RGB extraction and training utilities.
- Define a dedicated backbone-specific feature directory and index paths.

**How to interpret the result**
- If the paths exist and the extractor reports backbone support, the stronger RGB branch is ready to run.


In [ ]:
from pathlib import Path
import shutil

CODE_ROOT = Path('/content/CV_Image_pose_detection')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection')

SCALAR_TRAINER_REL = Path('artifacts/3_Modeling/train_pose_count_tcn.py')
RGB_EXTRACT_REL = Path('artifacts/3_Modeling/extract_rgb_frame_features.py')
RGB_TRAIN_REL = Path('artifacts/3_Modeling/train_rgb_count_tcn.py')
COMPARE_REL = Path('artifacts/3_Modeling/compare_count_run_to_baseline.py')

def sync_drive_file(rel: Path) -> None:
    src = CODE_ROOT / rel
    dst = DRIVE_PROJECT_ROOT / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not src.exists() and not dst.exists():
        print(f'[sync] missing both copies: {rel}')
        return
    if not src.exists():
        print(f'[sync] keeping Drive copy (no /content source): {rel}')
        return
    if not dst.exists():
        shutil.copy2(src, dst)
        print(f'[sync] copied /content -> Drive (Drive missing): {rel}')
        return
    if src.stat().st_mtime > dst.stat().st_mtime + 1.0:
        shutil.copy2(src, dst)
        print(f'[sync] copied newer /content -> Drive: {rel}')
    else:
        print(f'[sync] keeping Drive copy (newer or equal): {rel}')

for rel in [SCALAR_TRAINER_REL, RGB_EXTRACT_REL, RGB_TRAIN_REL, COMPARE_REL]:
    sync_drive_file(rel)

ANNOTATION_DIR = DRIVE_PROJECT_ROOT / 'Data/LLSP/annotation_cleaned'
VIDEO_DIR = DRIVE_PROJECT_ROOT / 'Data/LLSP/video'
POSE_INDEX = ANNOTATION_DIR / 'pose_feature_index.csv'

RGB_BACKBONE = 'resnet50'
RGB_FEATURE_DIR = ANNOTATION_DIR / f'rgb_{RGB_BACKBONE}_features'
RGB_INDEX = ANNOTATION_DIR / f'rgb_feature_index_{RGB_BACKBONE}_selected.csv'
RGB_SUMMARY = ANNOTATION_DIR / f'rgb_feature_summary_{RGB_BACKBONE}_selected.csv'

print('POSE_INDEX =', POSE_INDEX)
print('VIDEO_DIR =', VIDEO_DIR)
print('RGB_FEATURE_DIR =', RGB_FEATURE_DIR)
print('RGB extractor supports resnet50 =', 'resnet50' in (DRIVE_PROJECT_ROOT / RGB_EXTRACT_REL).read_text())


## Controlled Subset And Stronger RGB Preset

**Why this section exists**
- The stronger RGB branch should stay directly comparable to Stage 7.
- The main change should be the visual backbone, not the exercise scope.

**Approach**
- Reuse the same subset: `squat`, `pull_up`, `push_up`.
- Reuse the best pose `seq_len` from `6B`.
- Use lower extraction pressure and a slightly stronger RGB TCN capacity.

**How to interpret the result**
- If Stage 7 RGB improves under this stronger preset, the representation was the main weakness.


In [ ]:
import pandas as pd

TARGET_EXERCISES = ['squat', 'pull_up', 'push_up']
POSE_SEQ_LENS = {
    'squat': 256,
    'pull_up': 192,
    'push_up': 128,
}
RGB_EXTRACT_MAX_FRAMES = 128
RGB_EXTRACT_BATCH_SIZE = 8
RGB_TRAIN_CHANNELS = 128
RGB_TRAIN_NUM_BLOCKS = 5
RGB_TRAIN_DROPOUT = 0.25

meta_df = pd.read_csv(POSE_INDEX)
subset_counts = meta_df[meta_df['type'].isin(TARGET_EXERCISES)].groupby(['type', 'split']).size().unstack(fill_value=0).sort_index()
display(subset_counts)
print('RGB_BACKBONE =', RGB_BACKBONE)
print('POSE_SEQ_LENS =', POSE_SEQ_LENS)


## Stronger RGB Feature Extraction

**Why this section exists**
- The TCN trainer expects frozen RGB feature sequences, not raw videos.
- ResNet50 is stronger than ResNet18, but also more expensive, so extraction settings are reduced for stability.

**Approach**
- Extract `resnet50` features.
- Use `max_frames=128` and `batch_size=8` to reduce memory pressure.
- Keep live progress logging on.

**How to interpret the result**
- All `ok` rows mean the stronger RGB representation is ready for training.


In [ ]:
import subprocess

extract_cmd = [
    'python', '-u', str(DRIVE_PROJECT_ROOT / RGB_EXTRACT_REL),
    '--index-csv', str(POSE_INDEX),
    '--video-dir', str(VIDEO_DIR),
    '--feature-dir', str(RGB_FEATURE_DIR),
    '--output-index-csv', str(RGB_INDEX),
    '--output-summary-csv', str(RGB_SUMMARY),
    '--backbone', RGB_BACKBONE,
    '--max-frames', str(RGB_EXTRACT_MAX_FRAMES),
    '--batch-size', str(RGB_EXTRACT_BATCH_SIZE),
    '--device', 'cuda',
    '--overwrite',
    '--log-every', '1',
    '--save-progress-every', '0',
]
for exercise in TARGET_EXERCISES:
    extract_cmd.extend(['--exercise', exercise])

print('Running:', ' '.join(extract_cmd))
subprocess.run(extract_cmd, check=True)


In [ ]:
rgb_summary_df = pd.read_csv(RGB_SUMMARY)
display(rgb_summary_df['status'].value_counts())
display(rgb_summary_df.groupby(['type', 'status']).size().unstack(fill_value=0).sort_index())
display(rgb_summary_df.groupby('type')[['frames_total', 'frames_used', 'feature_dim']].mean().round(2))


## Stronger RGB TCN Training

**Why this section exists**
- This keeps the counting formulation fixed while giving RGB a stronger representation and slightly stronger temporal capacity.

**Approach**
- Train one RGB TCN per exercise.
- Use the stronger backbone features and a slightly wider TCN.
- Reuse the same `seq_len` settings as pose `6B`.

**How to interpret the result**
- A better result than Stage 7 RGB suggests representation strength mattered.


In [ ]:
import subprocess
import pandas as pd

RGB_RUNS = [
    {
        'exercise': 'squat',
        'seq_len': 256,
        'pose_run': 'pose_count_tcn_squat_seq256',
        'pose_best_run': 'squat_tcn_l1_channels96',
        'rgb_stage7_run': 'rgb_count_tcn_squat_seq256',
        'rgb_stronger_run': 'rgb_resnet50_count_tcn_squat_seq256',
    },
    {
        'exercise': 'pull_up',
        'seq_len': 192,
        'pose_run': 'pose_count_tcn_pull_up_seq192',
        'rgb_stage7_run': 'rgb_count_tcn_pull_up_seq192',
        'rgb_stronger_run': 'rgb_resnet50_count_tcn_pull_up_seq192',
    },
    {
        'exercise': 'push_up',
        'seq_len': 128,
        'pose_run': 'pose_count_tcn_push_up_seq128',
        'rgb_stage7_run': 'rgb_count_tcn_push_up_seq128',
        'rgb_stronger_run': 'rgb_resnet50_count_tcn_push_up_seq128',
    },
]

training_failures = []
for cfg in RGB_RUNS:
    cmd = [
        'python', str(DRIVE_PROJECT_ROOT / RGB_TRAIN_REL),
        '--project-dir', str(DRIVE_PROJECT_ROOT),
        '--index-csv', str(RGB_INDEX),
        '--run-name', cfg['rgb_stronger_run'],
        '--exercise', cfg['exercise'],
        '--seq-len', str(cfg['seq_len']),
        '--epochs', '80',
        '--batch-size', '16',
        '--lr', '0.001',
        '--weight-decay', '0.0001',
        '--channels', str(RGB_TRAIN_CHANNELS),
        '--kernel-size', '3',
        '--num-blocks', str(RGB_TRAIN_NUM_BLOCKS),
        '--dropout', str(RGB_TRAIN_DROPOUT),
        '--patience', '15',
        '--loss', 'l1',
        '--eval-transform', 'raw',
        '--selection-metric', 'mae',
        '--sampler', 'balanced_count',
        '--time-warp-range', '0.12',
        '--feature-noise-std', '0.02',
        '--frame-dropout-prob', '0.03',
        '--device', 'cuda',
    ]
    print('\nRunning:', ' '.join(cmd))
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as exc:
        training_failures.append({
            'exercise': cfg['exercise'],
            'run_name': cfg['rgb_stronger_run'],
            'returncode': exc.returncode,
        })
        print(f"FAILED: {cfg['exercise']} (returncode={exc.returncode})")

if training_failures:
    display(pd.DataFrame(training_failures))
else:
    print('All stronger RGB runs completed.')


## Pose vs Stage 7 RGB vs Stronger RGB Review

**Why this section exists**
- The goal is to isolate whether a stronger RGB representation actually improves on the Stage 7 RGB baseline.

**Approach**
- Load metrics for pose `6B`, Stage 7 RGB, and the stronger RGB run.
- Compare `valid_mae`, `valid_rmse`, and `valid_within_1` side by side.

**How to interpret the result**
- The strongest RGB branch should at least beat Stage 7 RGB if the backbone change mattered.


In [ ]:
import json
import pandas as pd

rows = []
for cfg in RGB_RUNS:
    variants = [
        ('pose_6B', cfg['pose_run']),
        ('rgb_stage7', cfg['rgb_stage7_run']),
        ('rgb_stronger', cfg['rgb_stronger_run']),
    ]
    if cfg.get('pose_best_run'):
        variants.insert(1, ('pose_best_squat', cfg['pose_best_run']))
    for variant, run_name in variants:
        metrics_path = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs' / run_name / 'metrics_summary.json'
        if not metrics_path.exists():
            continue
        with open(metrics_path, 'r', encoding='utf-8') as f:
            metrics = json.load(f)
        rows.append({
            'exercise': cfg['exercise'],
            'seq_len': cfg['seq_len'],
            'variant': variant,
            'run_name': run_name,
            'best_epoch': metrics.get('best_epoch'),
            'valid_mae': metrics['valid_metrics']['mae'],
            'valid_rmse': metrics['valid_metrics']['rmse'],
            'valid_within_1': metrics['valid_metrics']['within_1'],
        })

compare_df = pd.DataFrame(rows)
display(compare_df.sort_values(['exercise', 'variant']))

pivot_df = compare_df.pivot(index=['exercise', 'seq_len'], columns='variant', values=['valid_mae', 'valid_within_1'])
pivot_df.columns = ['_'.join(col).strip() for col in pivot_df.columns.values]
pivot_df = pivot_df.reset_index()
if 'valid_mae_rgb_stage7' in pivot_df.columns and 'valid_mae_rgb_stronger' in pivot_df.columns:
    pivot_df['delta_mae_stronger_minus_stage7'] = pivot_df['valid_mae_rgb_stronger'] - pivot_df['valid_mae_rgb_stage7']
if 'valid_within_1_rgb_stage7' in pivot_df.columns and 'valid_within_1_rgb_stronger' in pivot_df.columns:
    pivot_df['delta_within_1_stronger_minus_stage7'] = pivot_df['valid_within_1_rgb_stronger'] - pivot_df['valid_within_1_rgb_stage7']
if 'valid_mae_pose_best_squat' in pivot_df.columns and 'valid_mae_rgb_stronger' in pivot_df.columns:
    pivot_df['delta_mae_stronger_minus_best_squat'] = pivot_df['valid_mae_rgb_stronger'] - pivot_df['valid_mae_pose_best_squat']
if 'valid_within_1_pose_best_squat' in pivot_df.columns and 'valid_within_1_rgb_stronger' in pivot_df.columns:
    pivot_df['delta_within_1_stronger_minus_best_squat'] = pivot_df['valid_within_1_rgb_stronger'] - pivot_df['valid_within_1_pose_best_squat']
display(pivot_df.sort_values('exercise'))


## Stronger RGB Baseline Comparison Review

**Why this section exists**
- A stronger RGB model is only useful if it still beats the trivial train-split baseline.

**Approach**
- Compare the stronger RGB predictions against the trivial baseline.
- Then line them up beside pose `6B` and Stage 7 RGB.

**How to interpret the result**
- A stronger RGB run that improves over Stage 7 but not over the trivial baseline is still not convincing.


In [ ]:
import subprocess
import json
import pandas as pd

comparison_failures = []
for cfg in RGB_RUNS:
    run_dir = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs' / cfg['rgb_stronger_run']
    pred_path = run_dir / 'predictions.csv'
    if not pred_path.exists():
        continue
    summary_path = run_dir / 'baseline_comparison_summary.json'
    if summary_path.exists():
        continue
    cmd = [
        'python', str(DRIVE_PROJECT_ROOT / COMPARE_REL),
        '--index-csv', str(RGB_INDEX),
        '--predictions-csv', str(pred_path),
        '--exercise', cfg['exercise'],
    ]
    print('\nComparing:', ' '.join(cmd))
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as exc:
        comparison_failures.append({
            'exercise': cfg['exercise'],
            'run_name': cfg['rgb_stronger_run'],
            'returncode': exc.returncode,
        })

rows = []
for cfg in RGB_RUNS:
    for variant, run_name, index_path in [
        ('pose_6B', cfg['pose_run'], ANNOTATION_DIR / 'pose_sequence_index.csv'),
        ('rgb_stage7', cfg['rgb_stage7_run'], ANNOTATION_DIR / 'rgb_feature_index_selected.csv'),
        ('rgb_stronger', cfg['rgb_stronger_run'], RGB_INDEX),
    ]:
        summary_path = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs' / run_name / 'baseline_comparison_summary.json'
        if not summary_path.exists():
            continue
        with open(summary_path, 'r', encoding='utf-8') as f:
            summary = json.load(f)
        rows.append({
            'exercise': cfg['exercise'],
            'seq_len': cfg['seq_len'],
            'variant': variant,
            'run_name': run_name,
            'model_mae': summary['model_metrics']['mae'],
            'baseline_mae': summary['baseline_metrics']['mae'],
            'delta_mae_vs_trivial': summary['delta_vs_baseline']['mae'],
            'model_within_1': summary['model_metrics']['within_1'],
            'baseline_within_1': summary['baseline_metrics']['within_1'],
            'delta_within_1_vs_trivial': summary['delta_vs_baseline']['within_1'],
            'model_beats_baseline_rows': summary['row_level']['model_beats_baseline'],
            'valid_rows': summary['row_level']['valid_rows'],
        })

baseline_df = pd.DataFrame(rows)
display(baseline_df.sort_values(['exercise', 'variant']))

pivot_df = baseline_df.pivot(index=['exercise', 'seq_len'], columns='variant', values=['model_mae', 'model_within_1', 'delta_mae_vs_trivial', 'delta_within_1_vs_trivial'])
pivot_df.columns = ['_'.join(col).strip() for col in pivot_df.columns.values]
pivot_df = pivot_df.reset_index()
if 'model_mae_rgb_stage7' in pivot_df.columns and 'model_mae_rgb_stronger' in pivot_df.columns:
    pivot_df['delta_mae_stronger_minus_stage7'] = pivot_df['model_mae_rgb_stronger'] - pivot_df['model_mae_rgb_stage7']
if 'model_within_1_rgb_stage7' in pivot_df.columns and 'model_within_1_rgb_stronger' in pivot_df.columns:
    pivot_df['delta_within_1_stronger_minus_stage7'] = pivot_df['model_within_1_rgb_stronger'] - pivot_df['model_within_1_rgb_stage7']
display(pivot_df.sort_values('exercise'))

if comparison_failures:
    display(pd.DataFrame(comparison_failures))
